# EMCCD Photon Counting and Correlation Pipeline

This notebook performs the following steps:

1. Reads EMCCD dark/noise data.
2. Computes background/bias correction.
3. Fits the EMCCD noise distribution.
4. Loads photon-exposed EMCCD data.
5. Fits the photon-counting model.
6. Generates a threshold frame for photon counting.
7. Converts raw EMCCD frames into photon-counted data.
8. Computes convolution/correlation maps from photon-counted frames.

## Required packages

- `sifparser==0.3.6`
- `cupy`
- NVIDIA CUDA Toolkit
- `numpy`
- `matplotlib`

In [ ]:
import gc
from glob import glob
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import EMCCD_GPU as emccd

## User Parameters

In [ ]:
# ============================================================
# User-defined paths and filenames
# ============================================================

Data_path = "C:/RawData/"
Analysis_path = Data_path + "Analysis/"

file_name = "EMCCD_Count_file"
file_type = ".sif"

noise_file = "Noise_file"

# Create analysis folder if it does not exist
Path(Analysis_path).mkdir(parents=True, exist_ok=True)

# Find all photon-exposed files matching the chosen prefix
files = np.array([
    x[len(Data_path):]
    for x in glob(Data_path + f"{file_name}*{file_type}")
])

print("Files found:")
for f in files:
    print("  ", f)

## Noise Fitting: Read Noise Data and Compute Correction

In [ ]:
# ============================================================
# Noise fitting: read noise data and compute correction
# ============================================================

emccd.echo_flag = True

# Read noise/dark SIF file
noise = emccd.readSif(
    path=Data_path,
    file=f"{noise_file}.sif"
)[0]

# Background/bias correction from dark data
Crr_xN, correction = emccd.Bg_Correction(noise)

# Visualize correction frame and corrected noise frame
emccd.Beam(correction)
emccd.Beam(Crr_xN)

# Free raw noise data
del noise
gc.collect()

## Noise Probability Distribution

In [ ]:
# ============================================================
# Generate probability distribution from corrected noise data
# ============================================================

x, px, offset = emccd.make_prob(
    Crr_xN,
    range=[-70, 500],
    show_plot=True,
    make_offset=True
)

plt.show()

## Save Correction Frame

In [ ]:
# ============================================================
# Save correction frame
# ============================================================

correction_file = Analysis_path + f"correction{noise_file[-1]}.npy"
np.save(correction_file, correction)

print(f"Correction saved to: {correction_file}")

## Fit Noise Model

In [ ]:
# ============================================================
# Fit EMCCD noise model
# ============================================================

emccd.echo_flag = False

noise_fit = emccd.Noisefit(
    px,
    x,
    weights=emccd.weight(px, precision=5),
    params=(10, 0.05, 1e-4, 0.005)
)

emccd.plot_fit(
    x,
    noise_fit,
    px,
    save_file_name=Analysis_path + f"{noise_file}",
    save_options={
        "fig": False,
        "parameters": True
    }
)

print("Noise fitting complete.")

## Load Correction and Noise Parameters

In [ ]:
# ============================================================
# Load saved correction and noise-fit parameters
# ============================================================

correction = np.load(
    Analysis_path + f"correction{noise_file[-1]}.npy"
)

p0 = np.append(
    [5000],
    emccd.load_params(
        Analysis_path,
        f"{noise_file}_params.json"
    )
)

print("Correction shape:", correction.shape)
print("Initial photon-fit parameters:", p0)

## Optional Special Case: Cropped Correction

Only run this cell if your photon-exposed data is cropped, for example `70 × 70`.

In [ ]:
# ============================================================
# Optional special case: cropped correction
# ============================================================
#
# Run this cell only when needed.
# Otherwise skip it.
# ============================================================

# correction = np.load(
#     Analysis_path + f"correction{noise_file[-2:]}.npy"
# )
#
# correction = correction[:70, :70]
#
# p0 = np.append(
#     [5000],
#     emccd.load_params(
#         Analysis_path,
#         f"{noise_file}_params.json"
#     )
# )
#
# print("Using cropped correction.")
# print("Correction shape:", correction.shape)
# print("Initial photon-fit parameters:", p0)

## Photon Fitting Using One Representative File

In [ ]:
# ============================================================
# Photon fitting and threshold-frame generation
# ============================================================

# Use first photon-exposed file for photon fitting
f = files[0]

print(f"Photon fitting using file: {f}")

# Read raw photon-exposed data
data = emccd.readSif(
    Data_path,
    f"{f}"
)[0]

# Apply background/bias correction
Crr_xC = emccd.Bg_Correction(
    data,
    correction=correction
)

# Generate beam probability map
emccd.Beam(
    Crr_xC,
    assign_prob=True
)

# Create photon-exposed probability distribution
xp, pxp, offp = emccd.make_prob(
    Crr_xC,
    range=[-70, 500],
    make_offset=True,
    show_plot=True
)

plt.show()

## Fit Photon Model

In [ ]:
# ============================================================
# Fit photon model
# ============================================================

emccd.echo_flag = False

photon_fit = emccd.Photonfit(
    pxp,
    xp,
    weights=emccd.weight(pxp, precision=5),
    params=p0,
    vary_flags={
        "mu": True,
        "sigma": False,
        "pCIC": False,
        "pser": False,
        "pc": True
    }
)

emccd.plot_fit(
    xp,
    photon_fit,
    pxp
)

params_dict = photon_fit.best_values

params = [
    params_dict["mu"],
    params_dict["sigma"],
    params_dict["pCIC"],
    params_dict["pser"],
    params_dict["pc"]
]

max_mean_photons = params[0] * np.amax(emccd.bProb)

print("Photon-fit parameters:")
print("  mu    =", params[0])
print("  sigma =", params[1])
print("  pCIC  =", params[2])
print("  pser  =", params[3])
print("  pc    =", params[4])
print("Max Mean Photons =", max_mean_photons)

## Create and Save Threshold Frame

In [ ]:
# ============================================================
# Create threshold frame
# ============================================================

m_vals = np.arange(
    0.01,
    max_mean_photons + 0.1,
    0.1
)

T_frame = emccd.create_T_frame(
    m_vals,
    params,
    frame_size=[
        emccd.bProb.shape[0],
        emccd.bProb.shape[1]
    ]
)

T_frame_file = Analysis_path + f"{f[:-4]}_T_frame.npy"
np.save(T_frame_file, T_frame)

print(f"Threshold frame saved to: {T_frame_file}")
print("T_frame shape:", T_frame.shape)

# Free memory
del data, Crr_xC
gc.collect()

## Optional: Load Saved Threshold Frame Later

Use this if you reopen the notebook and do not want to repeat photon fitting.

In [ ]:
# ============================================================
# Optional: load existing threshold frame
# ============================================================
#
# Uncomment and edit filename if needed.
# ============================================================

# T_frame = np.load(Analysis_path + "D235_T_frame.npy")
# print("Loaded T_frame shape:", T_frame.shape)

## Photon Counting and Convolution

This section uses `flip_later=False`, so it corresponds to the convolution-style operation.

In [ ]:
# ============================================================
# Photon reading and convolution
# ============================================================

gpu_on = False
max_photon_number = 10

for f in files:

    print(f"Photon counting and convolving file: {f}")

    cd = emccd.read_photons(
        f,
        Data_path = Data_path,
        correction=correction,
        T_frame=T_frame,
        max_photon_number=max_photon_number
    )

    sf, nf = emccd.corr(
        cd,
        flip_later=False,
        gpu_on=gpu_on
    )

    val = sf - nf

    plt.figure()
    plt.imshow(val)
    plt.colorbar()
    plt.title(f"Convolution result: {f}")
    plt.show()

    save_file = Analysis_path + f"conv{f[1:-4]}.npy"
    np.save(save_file, val)

    print(f"Saved convolution result to: {save_file}")

    del cd, sf, nf, val
    gc.collect()

## Photon Counting and Correlation

This section uses `flip_later=True`, so it corresponds to the correlation-style operation.

In [ ]:
# ============================================================
# Photon reading and correlation
# ============================================================

gpu_on = False
max_photon_number = 10

for f in files:

    print(f"Photon counting and correlating file: {f}")

    cd = emccd.read_photons(
        f,
        Data_path = Data_path,
        correction=correction,
        T_frame=T_frame,
        max_photon_number=max_photon_number
    )

    sf, nf = emccd.corr(
        cd,
        flip_later=True,
        gpu_on=gpu_on
    )

    val = sf - nf

    plt.figure()
    plt.imshow(val)
    plt.colorbar()
    plt.title(f"Correlation result: {f}")
    plt.show()

    save_file = Analysis_path + f"corr{f[1:-4]}.npy"
    np.save(save_file, val)

    print(f"Saved correlation result to: {save_file}")

    del cd, sf, nf, val
    gc.collect()